# Notebook 12: Error-Driven Augmentation

## Purpose
Evaluate classifier-specific error-driven augmentation — the most principled
augmentation strategy in this project. Unlike Notebooks 10-11 which used a
single shared synthetic pool, each classifier here receives augmentation
tailored to its own failure modes.

## Methodology
For each classifier (LR, MLP-1, MLP-2):
1. False positives on the training set were identified at the Notebook 09
   threshold (τ=0.69 for all three)
2. K-means clustering (k=200) applied to false positive embeddings —
   selects diverse, representative failure modes rather than redundant
   near-duplicate errors
3. Sentence closest to each centroid selected as generation seed
4. `Llama-3-8B-Instruct` (temperature=0.7) rewrites each seed as a
   genuine hedge — preserving topic and vocabulary but adding real
   epistemic uncertainty
5. Two variants generated per seed, each using a different hedging device
6. Filtered by cosine similarity to nearest real positive (τ=0.8) —
   same filter as Notebook 10

## Why Classifier-Specific?
Each model makes structurally different errors:
- LR: 5,083 false positives (7.3%) — over-predicts due to linear boundary
- MLP-1: 1,134 false positives (1.6%) — more selective, harder failure modes
- MLP-2: 1,473 false positives (2.1%) — intermediate error profile

Augmenting with a shared pool would mix error types across models.
Classifier-specific augmentation ensures each model is challenged on
its own blind spots.

## Generated Files
| Classifier | Seeds | Raw variants | Source |
|---|---|---|---|
| LR | 199 | 398 | `error_driven_lr_raw.parquet` |
| MLP-1 | 200 | ~400 | `error_driven_mlp1_raw.parquet` |
| MLP-2 | 200 | ~400 | `error_driven_mlp2_raw.parquet` |

## Filtering
Cosine similarity to nearest real positive at τ=0.8 — identical to
Notebook 10. Error-driven variants are genuine hedges (label=1) so
the same filter logic applies: retain candidates geometrically close
to the real positive manifold.

## Evaluation
- UMAP: one plot per classifier condition, filtered synthetics overlaid
- Metrics table: precision, recall, F1 vs Notebooks 09-11
- DET curves: all conditions overlaid
- Key question: does classifier-specific augmentation outperform the
  shared augmentation strategies from Notebooks 10-11?